# Final Assignment - Canada Housing Prices Analysis
### Group 14
#### Ei Htoo Khaing, Lio, James Nguyen
In this assignment, we will self-answer qeustions about Canadian real estate price trends and investigates how they are influenced by key socioeconomic factors.  
The dataset is retrieved from github [https://github.com/tahczeban/CANADIAN_REAL_ESTATE_PRICES].

We understand that, when submitting this assingment, we are abiding to Seneca's standards of academic honesty.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
                    .getOrCreate()

### 1: Data Cleaning and Preparation

First, open the files and set the variables. Clean the unnecessary columns or change columns.

In [41]:
from pyspark.sql.functions import col

house_df = spark.read.csv('/data/avg_house_price.csv', header=True, inferSchema=True)
pop_df = spark.read.csv('/data/population_analysis_sid.csv', header=True, inferSchema=True)
income_df = spark.read.csv('/data/prov_hh_income.csv', header=True, inferSchema=True)

# Remove null values if there's any, then filter out only the years needed to analyse
house_df = house_df.dropna().filter(col('Year') != 2021)
pop_df = pop_df.dropna().filter(col("Year") >= 2015).filter(col("Year") != 2021)
income_df = income_df.dropna().filter(col("Year") >= 2015)

In [55]:
income_df.filter(col('Year') == 2015).orderBy("Province").show()

+----+--------+------+
|Year|Province|Income|
+----+--------+------+
|2015| Alberta|108594|
|2015|      BC| 79234|
|2015|      MB| 72971|
|2015|      NB| 63767|
|2015|      NL| 75183|
|2015|      NS| 64717|
|2015|      NU|110546|
|2015|     NWT|116295|
|2015|     ONT| 79674|
|2015|     PEI| 63660|
|2015|      QC| 61161|
|2015|      SK| 83814|
|2015|      YK| 96529|
+----+--------+------+



In [56]:
pop_df.filter(col('Year') == 2015).orderBy("Province").show()

+----+--------+----------+
|Year|Province|Population|
+----+--------+----------+
|2015|      AB|  16548357|
|2015|      BC|  19082570|
|2015|      MB|   5164103|
|2015|      NB|   3037185|
|2015|      NL|   2113237|
|2015|      NS|   3749412|
|2015|      NU|    145593|
|2015|     NWT|    176998|
|2015|     ONT|  54792403|
|2015|     PEI|    578067|
|2015|      QC|  32691481|
|2015|      SK|   4482873|
|2015|      YK|    149981|
+----+--------+----------+



In [57]:
house_df.filter(col('Year') == 2015).orderBy("Province").show()

+----+--------+---------+
|Year|Province|Avg_price|
+----+--------+---------+
|2015|      AB|   432800|
|2015|      BC|   610000|
|2015|      MB|   283000|
|2015|      NB|   199800|
|2015|      NL|   299300|
|2015|      NS|   256100|
|2015|     ONT|   455000|
|2015|     PEI|   204200|
|2015|      QC|   305800|
|2015|      SK|   332600|
+----+--------+---------+



In [65]:
# After checking the housing dataset, only 10 provinces' housing data are recorded. Therefore, I'll filter the "NU, NWT, YK" data from pop_df and income_df
# First, change alberta to AB
from pyspark.sql.functions import when, col

income_df = income_df.withColumn("Province",
    when(col("Province") == "Alberta", "AB")
    .otherwise(col("Province"))
)

#Filter the provinces needed
provinces = ["AB","BC","MB","NB","NL","NS","ONT","PEI","QC", "SK"]

pop_df = pop_df.filter(col("Province").isin(provinces))
income_df = income_df.filter(col("Province").isin(provinces))
pop_df.count()
income_df.count()
house_df.count()


60

In [67]:
# Join datasets
combined_df = house_df \
    .join(pop_df, on = ["Year", "Province"], how = "inner") \
    .join(income_df, on = ["Year", "Province"], how = "inner")

combined_df.show()

+----+--------+---------+----------+------+
|Year|Province|Avg_price|Population|Income|
+----+--------+---------+----------+------+
|2015|      AB|   432800|  16548357|108594|
|2016|      AB|   425900|  16756614| 99059|
|2017|      AB|   429800|  16945119|102077|
|2018|      AB|   428500|  17166499|101717|
|2019|      AB|   420900|  17422845|104157|
|2020|      AB|   420900|  17661466|107480|
|2015|      BC|   610000|  19082570| 79234|
|2016|      BC|   717600|  19386872| 79996|
|2017|      BC|   775400|  19681992| 83217|
|2018|      BC|   816300|  20001433| 82730|
|2019|      BC|   785500|  20339311| 87208|
|2020|      BC|   848200|  20603068| 97537|
|2015|      MB|   283000|   5164103| 72971|
|2016|      MB|   288800|   5244283| 72772|
|2017|      MB|   297400|   5327896| 75835|
|2018|      MB|   300000|   5402266| 74748|
|2019|      MB|   305200|   5470522| 76033|
|2020|      MB|   314400|   5518408| 82483|
|2015|      NB|   199800|   3037185| 63767|
|2016|      NB|   207800|   3051

In [85]:
from pyspark.sql.functions import avg

price_by_year = combined_df.groupBy("Year")\
    .agg(avg("Avg_price").alias("Avg_House_Price"))\
        .orderBy("Year")

price_by_year.show()

+----+---------------+
|Year|Avg_House_Price|
+----+---------------+
|2015|       337860.0|
|2016|       357930.0|
|2017|       377630.0|
|2018|       386070.0|
|2019|       387430.0|
|2020|       412000.0|
+----+---------------+



In [84]:
income_by_year = combined_df.groupBy("Year")\
    .agg(avg("Income").alias("Avg_Income"))\
        .orderBy("Year")
income_by_year.show()

+----+----------+
|Year|Avg_Income|
+----+----------+
|2015|   75277.5|
|2016|   74315.8|
|2017|   76475.4|
|2018|   76648.4|
|2019|   78548.9|
|2020|   84630.2|
+----+----------+



In [78]:
pop_by_year = combined_df.groupBy("Year")\
    .agg(avg("Population").alias("Avg_Population"))\
        .orderBy("Year")
pop_by_year.show()

+----+--------------+
|Year|Avg_Population|
+----+--------------+
|2015|  1.42239688E7|
|2016|  1.43730751E7|
|2017|  1.45491937E7|
|2018|  1.47518737E7|
|2019|  1.49661326E7|
|2020|  1.51473592E7|
+----+--------------+



In [87]:
# Join datasets
combined_avg_df = price_by_year \
    .join(pop_by_year, on = "Year", how = "inner") \
    .join(income_by_year, on = "Year", how = "inner")

combined_avg_df.show()

+----+---------------+--------------+----------+
|Year|Avg_House_Price|Avg_Population|Avg_Income|
+----+---------------+--------------+----------+
|2018|       386070.0|  1.47518737E7|   76648.4|
|2015|       337860.0|  1.42239688E7|   75277.5|
|2019|       387430.0|  1.49661326E7|   78548.9|
|2020|       412000.0|  1.51473592E7|   84630.2|
|2016|       357930.0|  1.43730751E7|   74315.8|
|2017|       377630.0|  1.45491937E7|   76475.4|
+----+---------------+--------------+----------+

